# Dhaka PM2.5 — Part 1: Explore the data

We want to predict how polluted Dhaka's air will be in the next hour.

**This notebook only looks. It decides nothing and changes nothing.**

That order is deliberate. If we clean first, we clean based on assumptions.
If we look first, every later decision — which columns to keep, which to throw
away, how to split, which model — is backed by something we actually saw.

At the end there is a list of the decisions this exploration opens up. We make
those in Part 2.

> **Runtime:** use **CPU** (Runtime -> Change runtime type -> CPU).
> Spark's ML library has no GPU support, so a GPU here does nothing.

Every code cell has a description above it explaining what it does and what to
look for in the output.

---
# Setup

## Step 1 — Check Python and Java

PySpark is only a thin Python wrapper. The real work happens in a **Java**
program running beside your notebook. So both have to be present.

This cell prints versions. It changes nothing.

In [ ]:
import sys
import subprocess

print("Python:", sys.version.split()[0])

# Java writes its version to stderr rather than stdout, hence .stderr below
java = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java  :", java.stderr.strip().splitlines()[0] if java.stderr else "NOT FOUND")

## Step 2 — Install PySpark

About 300 MB, so we skip it if it is already installed.

Version **4.0.4** is chosen deliberately: Colab now runs Python 3.13, which the
older PySpark 3.5 does not support, and Colab preinstalls a package that expects
PySpark 4.0.x. Anything else produces either a crash or a conflict warning.

Takes 1-2 minutes the first time.

In [ ]:
try:
    import pyspark
    print("PySpark already installed:", pyspark.__version__)
except ImportError:
    print("Installing PySpark, this takes a minute...")
    !pip install -q pyspark==4.0.4
    import pyspark
    print("Installed PySpark:", pyspark.__version__)

## Step 3 — Download the data

The readings come from the **US Embassy in Dhaka**. It has measured PM2.5 every
hour since 2016 and publishes the results through the AirNow programme.

One CSV per year, ten files, about 8 MB in total. The data is public — no
account, no API key.

Note: most tutorials point at `dosairnowdata.org`, which no longer exists. The
address below is the live storage that the EPA's own embassy map reads from.

Re-running this cell is free — it skips files already downloaded.

In [ ]:
import os
import urllib.request

BASE_URL = ("https://s3-us-west-1.amazonaws.com/files.airnowtech.org"
            "/airnow/EmbassyHistorical/Dhaka")
FOLDER = "data"

os.makedirs(FOLDER, exist_ok=True)

for year in range(2016, 2026):
    filename = f"Dhaka_PM2.5_{year}_YTD.csv"
    filepath = os.path.join(FOLDER, filename)

    if os.path.exists(filepath):
        print("already have:", filename)
    else:
        urllib.request.urlretrieve(f"{BASE_URL}/{year}/{filename}", filepath)
        print("downloaded  :", filename)

## Step 4 — Read the raw text before touching Spark

Always look at the actual characters in the file first. It takes ten seconds and
prevents a long chain of wrong assumptions.

**Look for three things in the output:**

1. Column names contain spaces and dots — `Raw Conc.`, `QC Name`. Awkward in
   Spark, so in Step 6 we give them clean names.
2. The date is written in **12-hour** form with `AM`/`PM`.
3. The first data rows say `-999.0` and `Missing`. That is the station's code
   for *no reading*. Not a measurement.

In [ ]:
with open("data/Dhaka_PM2.5_2016_YTD.csv") as f:
    for i in range(4):
        print(f.readline().strip())
        print()

## Step 5 — Start Spark

A `SparkSession` is the single entry point to everything in Spark.

| Setting | Meaning |
|---|---|
| `master("local[*]")` | Run here, use every CPU core |
| `appName(...)` | A label that only shows up in logs |
| `timeZone("Asia/Dhaka")` | The readings are Dhaka local time |

First run takes ~20 seconds while the Java process boots.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("dhaka-pm25-explore")
         .master("local[*]")
         .config("spark.sql.session.timeZone", "Asia/Dhaka")
         .getOrCreate())

# Spark is very chatty by default. Show real errors only.
spark.sparkContext.setLogLevel("ERROR")

print("Spark running, version", spark.version)

## Step 6 — Tell Spark the column types

Spark can guess types by scanning the file (`inferSchema=True`). We write them
out by hand instead, for two reasons:

1. Guessing means reading all the data twice.
2. The streaming part of this project later **refuses** to guess. So this has to
   be written eventually — better once, now.

A useful side effect: because we supply the names, Spark ignores the messy
header in the file and uses ours. No renaming step needed later.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType,
)

# Order MUST match the column order in the CSV.
schema = StructType([
    StructField("site",         StringType()),    # expect: always "Dhaka"
    StructField("parameter",    StringType()),    # expect: always "PM2.5 - Principal"
    StructField("date_text",    StringType()),    # e.g. "2016-01-01 01:00 AM"
    StructField("year",         IntegerType()),
    StructField("month",        IntegerType()),
    StructField("day",          IntegerType()),
    StructField("hour",         IntegerType()),   # 0 to 23
    StructField("nowcast",      DoubleType()),    # AirNow's smoothed value
    StructField("aqi",          IntegerType()),   # Air Quality Index, 0-500
    StructField("aqi_category", StringType()),    # "Good" ... "Hazardous"
    StructField("pm25",         DoubleType()),    # the reading we care about
    StructField("unit",         StringType()),    # expect: always "UG/M3"
    StructField("duration",     StringType()),    # expect: always "1 Hr"
    StructField("qc",           StringType()),    # "Valid" / "Missing" / ...
])

print(len(schema.fields), "columns defined")

## Step 7 — Read all ten files as one table

Point Spark at the **folder**, not a file, and it treats every CSV inside as one
table.

`.cache()` keeps the result in memory. Without it, every question we ask below
would re-read all ten files from disk.

In [ ]:
raw = spark.read.option("header", True).schema(schema).csv("data").cache()

print("rows:", f"{raw.count():,}")
raw.show(5)

---
# Explore

From here on we only ask questions. Nothing is modified.

## Q1 — Which columns actually carry information?

A column holding the same value in every row tells the model nothing. Four
columns look like that. Rather than assume it, count the distinct values.

**Look for:** which columns come back with exactly one value.

In [ ]:
print(f"{'column':<15}{'distinct values':>16}   sample")
print("-" * 60)

for column in raw.columns:
    n = raw.select(column).distinct().count()
    sample = raw.select(column).first()[0]
    print(f"{column:<15}{n:>16,}   {sample!r}")

## Q2 — What does the quality flag say?

Every row carries a `qc` flag written by the monitoring station.

**Look for:** how many rows are *not* `Valid`. Those rows hold `-999` instead of
a reading — the station's code for "no data". `-999` is not a small error; it is
further from a normal reading than the worst pollution ever recorded here, so it
would wreck any model trained on it.

In [ ]:
raw.groupBy("qc").count().orderBy("count", ascending=False).show()

### What a bad row actually looks like

Seeing one makes the point better than the counts do.

In [ ]:
(raw.filter(F.col("qc") != "Valid")
    .select("date_text", "pm25", "nowcast", "aqi", "aqi_category", "qc")
    .show(5, truncate=False))

## Build a working table

To explore further we need two small things: drop the no-data rows, and turn the
four separate time columns into one real timestamp.

This is *not* the cleaning step — that is Part 2, and it will be based on what we
find here. This is only enough to make the questions below answerable.

We build the timestamp from `year`/`month`/`day`/`hour` rather than parsing the
text, because reading four numbers cannot trip over AM/PM.

In [ ]:
work = (raw
        .filter(F.col("qc") == "Valid")
        .filter(F.col("pm25") != -999.0)
        .withColumn("ts", F.expr("make_timestamp(`year`, `month`, `day`, `hour`, 0, 0)"))
        .cache())

print("rows kept:", f"{work.count():,}")
work.select("ts", "hour", "pm25", "nowcast", "aqi", "aqi_category").show(5)

## Q3 — How much data is there, and over what period?

**Look for:** the coverage percentage per year. A year at 74% has three months
missing somewhere. The last year is partial — the station stopped reporting.

In [ ]:
first_ts, last_ts = work.agg(F.min("ts"), F.max("ts")).first()
print("first reading:", first_ts)
print("last reading :", last_ts)
print()

per_year = (work.groupBy("year").count().orderBy("year").collect())
for row in per_year:
    bar = "#" * int(row["count"] / 200)
    print(f"  {row['year']}  {row['count']:>6,}  {row['count'] / 8760:>5.0%}  {bar}")

## Q4 — How many hours are missing?

The station did not record every hour. This matters more than it sounds.

Later we will build inputs like *"the reading one hour ago"*. Spark's `lag()`
looks **one row back, not one hour back**. If an hour is missing, those two are
different things, and the model silently learns from a reading that might be
days old.

**Look for:** the size of the missing percentage.

In [ ]:
hours_expected = int((last_ts - first_ts).total_seconds() / 3600) + 1
hours_present = work.count()
hours_missing = hours_expected - hours_present

print("hours expected:", f"{hours_expected:,}")
print("hours present :", f"{hours_present:,}")
print("hours MISSING :", f"{hours_missing:,}",
      f"({hours_missing / hours_expected:.1%})")

### Are the gaps many small ones, or a few huge ones?

The answer changes what we should do about them. A thousand one-hour blips is a
very different problem from one three-month outage.

We sort by time, look at the gap to the previous row, and group the results.

**Look for:** how many gaps are just 1 hour, versus how long the worst one is.

In [ ]:
from pyspark.sql.window import Window

by_time = Window.orderBy("ts")

gaps = (work
        .select("ts")
        .withColumn("previous_ts", F.lag("ts").over(by_time))
        # difference in hours between this reading and the one before it
        .withColumn("gap_hours",
                    (F.col("ts").cast("long") - F.col("previous_ts").cast("long")) / 3600)
        .filter(F.col("gap_hours") > 1)
        .withColumn("missing_hours", F.col("gap_hours") - 1))

print("number of separate gaps:", f"{gaps.count():,}")
print()

(gaps.withColumn("size",
                 F.when(F.col("missing_hours") == 1, "1 hour")
                  .when(F.col("missing_hours") <= 3, "2-3 hours")
                  .when(F.col("missing_hours") <= 12, "4-12 hours")
                  .when(F.col("missing_hours") <= 24, "13-24 hours")
                  .when(F.col("missing_hours") <= 168, "1-7 days")
                  .otherwise("more than a week"))
     .groupBy("size").count().orderBy("count", ascending=False).show(truncate=False))

worst = gaps.agg(F.max("missing_hours")).first()[0]
print(f"largest single gap: {worst:,.0f} hours = {worst / 24:.0f} days")

## Q5 — What do the readings look like?

`describe()` gives count, mean, spread, min and max.

**Look for:** the mean against the World Health Organization guideline of
**15 ug/m3** for a 24-hour average, and how far the maximum sits above the mean.

In [ ]:
work.select("pm25").describe().show()

### Percentiles say more than the mean

The mean of a spiky series is misleading. Percentiles show the actual shape:
`p50` is the middle reading, `p99` means only 1% of hours were worse.

In [ ]:
levels = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
values = work.approxQuantile("pm25", levels, 0.001)

for level, value in zip(levels, values):
    print(f"  p{int(level * 100):<3} {value:>7.1f} ug/m3   {'#' * int(value / 8)}")

### Draw the distribution

A histogram makes the shape obvious in a way numbers do not.

`toPandas()` copies Spark data into ordinary Python memory. Safe here — one
column of 75,000 numbers is tiny — but never do it on a large table.

**Look for:** a long tail stretching right. Most hours are moderate, a few are
extreme. Statisticians call this *right-skewed*.

In [ ]:
import matplotlib.pyplot as plt

readings = work.select("pm25").toPandas()["pm25"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(readings, bins=80, color="steelblue")
axes[0].axvline(15, color="green", linestyle="--", label="WHO guideline (15)")
axes[0].set_title("PM2.5 readings")
axes[0].set_xlabel("ug/m3")
axes[0].legend()

# log scale on the y axis, so the rare extreme hours stay visible
axes[1].hist(readings, bins=80, color="indianred")
axes[1].set_yscale("log")
axes[1].set_title("same data, log scale on y axis")
axes[1].set_xlabel("ug/m3")

plt.tight_layout()
plt.show()

### Would a log transform help?

Many models prefer a roughly symmetric target. **Skew** measures lopsidedness:
0 is symmetric, above +1 is strongly stretched to the right.

We compare the raw readings with `log(1 + reading)`.

**Look for:** whether the log version is much closer to 0. If it is, predicting
the log and converting back afterwards becomes a genuine option for Part 3.

In [ ]:
import numpy as np

def skew(values):
    values = np.asarray(values, dtype=float)
    centred = values - values.mean()
    return (centred ** 3).mean() / (values.std() ** 3)

print(f"skew of raw readings   : {skew(readings):+.3f}")
print(f"skew of log(1+reading) : {skew(np.log1p(readings)):+.3f}")

## Q6 — Does pollution follow the seasons?

Dhaka has a dry season and a monsoon. Rain washes particles out of the air, so
we would expect a difference. The question is how big.

**Look for:** the ratio between the worst month and the best month.

In [ ]:
monthly = (work.groupBy("month")
               .agg(F.avg("pm25").alias("average"),
                    F.count("*").alias("hours"))
               .orderBy("month")
               .toPandas())

NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
         "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

for _, row in monthly.iterrows():
    name = NAMES[int(row["month"]) - 1]
    print(f"  {name}  {row['average']:>6.1f}  {'#' * int(row['average'] / 4)}")

print()
print(f"worst month is {monthly['average'].max() / monthly['average'].min():.1f}x "
      f"the cleanest month")

In [ ]:
plt.figure(figsize=(11, 4))
plt.bar(NAMES, monthly["average"], color="steelblue")
plt.axhline(15, color="green", linestyle="--", label="WHO guideline (15)")
plt.ylabel("average PM2.5 (ug/m3)")
plt.title("Dhaka PM2.5 by month, 2016-2025")
plt.legend()
plt.tight_layout()
plt.show()

## Q7 — Does pollution follow the time of day?

**Look for:** a dip in the afternoon and a peak late at night. The physical
reason is that afternoon sun heats the ground, air rises and mixes, and the
pollution spreads through a taller column. At night the air settles into a thin
layer near the ground and the same emissions get concentrated.

In [ ]:
hourly = (work.groupBy("hour")
              .agg(F.avg("pm25").alias("average"))
              .orderBy("hour")
              .toPandas())

for _, row in hourly.iterrows():
    print(f"  {int(row['hour']):>2}:00  {row['average']:>6.1f}  "
          f"{'#' * int(row['average'] / 4)}")

plt.figure(figsize=(11, 4))
plt.plot(hourly["hour"], hourly["average"], marker="o")
plt.xticks(range(0, 24, 2))
plt.xlabel("hour of day")
plt.ylabel("average PM2.5 (ug/m3)")
plt.title("Daily rhythm")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Q8 — Does the day of the week matter?

Traffic is lighter on the weekend, so it might.

**Look for:** whether the bars are actually different, or all about the same
height. A flat result is a useful finding — it means this column would be dead
weight as a model input.

In [ ]:
# dayofweek: 1 = Sunday ... 7 = Saturday
weekly = (work.withColumn("dow", F.dayofweek("ts"))
              .groupBy("dow")
              .agg(F.avg("pm25").alias("average"))
              .orderBy("dow")
              .toPandas())

DAYS = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]

for _, row in weekly.iterrows():
    print(f"  {DAYS[int(row['dow']) - 1]}  {row['average']:>6.1f}  "
          f"{'#' * int(row['average'] / 2)}")

spread = weekly["average"].max() - weekly["average"].min()
print()
print(f"difference between busiest and quietest day: {spread:.1f} ug/m3")
print(f"for comparison, the monthly swing was "
      f"{monthly['average'].max() - monthly['average'].min():.1f} ug/m3")

## Q9 — Is the air getting better or worse over the years?

**Look for:** the trend, but read it carefully. Check each year's coverage from
Q3 before believing its average. A year that only recorded the dry season will
look terrible for reasons that have nothing to do with pollution getting worse.

In [ ]:
yearly = (work.groupBy("year")
              .agg(F.avg("pm25").alias("average"),
                   F.count("*").alias("hours"))
              .orderBy("year")
              .toPandas())

for _, row in yearly.iterrows():
    coverage = row["hours"] / 8760
    flag = "   <-- partial year, do not compare" if coverage < 0.5 else ""
    print(f"  {int(row['year'])}  {row['average']:>6.1f}  "
          f"(coverage {coverage:>4.0%}){flag}")

## Q10 — How fast does the air change from one hour to the next?

This sets the difficulty of the whole problem. If pollution barely moves in an
hour, predicting the next hour is easy. If it swings wildly, it is hard.

We compare each reading with the one an hour before. Because of the gaps found
in Q4, we match on **the actual timestamp**, not on the previous row.

**Look for:** the standard deviation, and the share of hours that jump by more
than 50. That share is roughly how often a naive "next hour equals this hour"
guess would fail badly.

In [ ]:
now = work.select("ts", F.col("pm25").alias("now"))
one_hour_earlier = work.select(
    (F.col("ts") + F.expr("INTERVAL 1 HOUR")).alias("ts"),
    F.col("pm25").alias("before"))

changes = (now.join(one_hour_earlier, "ts")
              .withColumn("change", F.col("now") - F.col("before"))
              .cache())

changes.select("change").describe().show()

big_jumps = changes.filter(F.abs(F.col("change")) > 50).count()
print(f"hours that moved more than 50 ug/m3: {big_jumps:,} "
      f"({big_jumps / changes.count():.1%})")

## Q11 — How long does the air remember?

**The single most useful chart in this notebook.** It decides which inputs we
build in Part 2.

*Correlation* here means: if I know the reading N hours ago, how well does that
predict the reading now? 1.0 is perfect, 0.0 is useless.

We compute it by shifting the table forward N hours and joining it to itself on
the timestamp.

**Look for two things:**

1. How fast the line falls as N grows.
2. Whether it **rises again** near 24 hours. If it does, that is the daily cycle
   showing up — the same hour yesterday is a genuinely good clue, better than
   twelve hours ago.

In [ ]:
LAGS = [1, 2, 3, 6, 12, 18, 24, 48, 72, 168]
results = []

for lag in LAGS:
    shifted = work.select(
        (F.col("ts") + F.expr(f"INTERVAL {lag} HOURS")).alias("ts"),
        F.col("pm25").alias("past"))

    joined = work.select("ts", "pm25").join(shifted, "ts")
    r = joined.corr("pm25", "past")
    results.append((lag, r))

    print(f"  {lag:>4} hours ago   correlation {r:.3f}   {'#' * int(r * 50)}")

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot([lag for lag, _ in results], [r for _, r in results], marker="o")
plt.axvline(24, color="orange", linestyle="--", label="24 hours (same time yesterday)")
plt.xlabel("hours ago")
plt.ylabel("correlation with the current reading")
plt.title("How long the air remembers")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Q12 — Can we use `nowcast`, `aqi` and `aqi_category` as inputs?

This is the trap that quietly ruins air-quality projects, so it gets its own
section.

`nowcast` is AirNow's smoothed number. It is a weighted average of roughly the
last twelve hours — **including the hour we are trying to predict**. `aqi` and
`aqi_category` are then calculated from `nowcast`.

If we feed the model `nowcast` for the same hour, we are handing it the answer.
The model would score beautifully in testing and be useless in reality, because
in a live system that value does not exist yet.

**Look for:** how close the correlation is to 1.0.

In [ ]:
# A handful of rows are flagged Valid but still carry the -999 no-data code in
# nowcast and aqi. Left in, those few rows would drag the correlation down and
# tell us nothing. Worth knowing they exist -- a quality flag is not a guarantee.
hidden = work.filter(F.col("nowcast") == -999.0).count()
print(f"rows flagged Valid but holding -999 in nowcast: {hidden}")
print()

honest = work.filter(F.col("nowcast") != -999.0)

print("correlation with the reading we want to predict:")
print(f"  nowcast, same hour : {honest.corr('pm25', 'nowcast'):.4f}")
print(f"  aqi,     same hour : {honest.corr('pm25', 'aqi'):.4f}")

### A second piece of evidence

If `aqi_category` described the current reading, each category would occupy its
own band of values with no overlap.

**Look for:** the min and max columns. The bands overlap heavily — `Good` hours
and `Unhealthy` hours share the same raw readings. That is because the category
describes the smoothed twelve-hour value, not this hour.

In [ ]:
(work.groupBy("aqi_category")
     .agg(F.count("*").alias("hours"),
          F.min("pm25").alias("lowest"),
          F.avg("pm25").alias("average"),
          F.max("pm25").alias("highest"))
     .orderBy("average")
     .show(truncate=False))

## Q13 — How often is the air actually dangerous?

The official health scale, in plain counts. This is the number that makes the
project worth doing.

In [ ]:
total = work.count()

counts = (work.groupBy("aqi_category")
              .count()
              .orderBy("count", ascending=False)
              .collect())

for row in counts:
    share = row["count"] / total
    print(f"  {row['aqi_category']:<32} {row['count']:>7,}  {share:>6.1%}  "
          f"{'#' * int(share * 100)}")

## Q14 — What did the worst hours look like?

**Look for:** when they happened. If the extremes cluster in particular months,
that agrees with the seasonal pattern from Q6.

In [ ]:
(work.select("ts", "pm25", "aqi_category")
     .orderBy(F.col("pm25").desc())
     .show(10, truncate=False))

## Q15 — See it with your own eyes

Statistics summarise. A plot shows the texture: the daily rhythm, the multi-day
episodes, and how sharply spikes arrive.

In [ ]:
one_month = (work
             .filter((F.col("ts") >= "2024-01-01") & (F.col("ts") < "2024-02-01"))
             .orderBy("ts")
             .select("ts", "pm25")
             .toPandas())

plt.figure(figsize=(14, 4))
plt.plot(one_month["ts"], one_month["pm25"], linewidth=1)
plt.axhline(15, color="green", linestyle="--", label="WHO guideline (15)")
plt.ylabel("PM2.5 (ug/m3)")
plt.title("January 2024, hour by hour")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
one_year = (work
            .filter(F.col("year") == 2024)
            .orderBy("ts")
            .select("ts", "pm25")
            .toPandas())

plt.figure(figsize=(14, 4))
plt.plot(one_year["ts"], one_year["pm25"], linewidth=0.4)
plt.axhline(15, color="green", linestyle="--", label="WHO guideline (15)")
plt.ylabel("PM2.5 (ug/m3)")
plt.title("All of 2024 — the monsoon dip is visible in the middle")
plt.legend()
plt.tight_layout()
plt.show()

---
# What we found

Fill in the numbers from your own run — they should match these.

| # | Question | Finding |
|---|---|---|
| Q1 | Useful columns? | `site`, `parameter`, `unit`, `duration` never change. Dead weight. |
| Q2 | Quality flag? | ~2,300 rows are not `Valid` and hold `-999`. Must go. |
| Q3 | Coverage? | 2016-03 to 2025-03. Some years near 100%, one near 74%, the last only ~23%. |
| Q4 | Missing hours? | ~5% missing, in ~517 separate gaps. Most are 1 hour; the worst is ~76 days. |
| Q5 | Distribution? | Median ~65, mean ~93, max 985. Strongly right-skewed; `log` almost fixes it. |
| Q6 | Season? | January averages ~200, July ~31. A **6x** swing. The strongest pattern in the data. |
| Q7 | Time of day? | Peaks around midnight (~112), dips mid-afternoon (~63). |
| Q8 | Day of week? | Essentially flat — under 7 ug/m3 across all days. Weak signal. |
| Q9 | Yearly trend? | Drifts upward, but the last year is partial and dry-season-only. |
| Q10 | Hourly volatility? | Standard deviation ~33. About 7% of hours move more than 50. |
| Q11 | Memory? | 0.92 at 1 hour, falls to ~0.55 by 12 hours, **rises back to ~0.72 at 24 hours**. |
| Q12 | `nowcast` / `aqi`? | Correlate 0.97 / 0.84 with the target *for the same hour*. Contain the answer. |
| Q13 | Health? | Only ~2% of hours are `Good`. A third are `Unhealthy` or worse. |

# Decisions this opens up — for Part 2

**1. The constant columns.** Q1 settles it: drop them.

**2. The `-999` rows.** Q2 settles it: drop them. But note this leaves holes in
time, which feeds into the next decision.

**3. The missing hours.** Q4 says ~5%, mostly single hours. Options: build a
complete hourly timeline so that "one row back" always means "one hour back";
or fill the small gaps and cut around the large ones. The 76-day gap has to be
handled either way.

**4. `nowcast`, `aqi`, `aqi_category`.** Q12 says using them for the current
hour is cheating. Options: drop them entirely, or use only their **past** values.
Their high correlation makes the past values potentially useful.

**5. Which lags to build.** Q11 is the evidence. 1, 2 and 3 hours ago are
strong. 24 hours ago beats 12 hours ago, so the daily cycle is real and worth a
feature. Whether 168 hours (one week) earns its place is an open question — it
costs rows, because early hours have no week-old value to look back at.

**6. Calendar inputs.** Q6 makes month clearly worth including and Q7 makes hour
worth including. Q8 suggests day of week is not.

**7. Transform the target?** Q5 shows `log(1 + reading)` is far more symmetric.
Options: predict the raw value, or predict the log and convert back. Worth
testing both.

**8. Where to split train and test.** It must be by **time**, never randomly —
shuffling would let the model train on the future and be tested on the past.
The open question is the cut-off date, and Q3 and Q9 say to be careful: the
final partial year is dry-season-only, so a test set made of it alone is
seasonally biased.

**9. Which model.** With one station and one measurement, the useful inputs are
past readings and calendar position. Gradient-boosted trees suit that: they
handle skew and interactions without scaling, and they report which input
mattered. A linear model makes a good sanity check.

**10. Do we need PCA?** Reasonable to ask, and the honest answer here is
probably no.

PCA compresses many correlated columns into a few. We have the opposite
situation: after dropping the constants and the leaky columns, we are down to
about three real inputs plus whatever lags we build.

It becomes a fairer question *after* Part 2, because lag features are heavily
correlated with each other — Q11 shows exactly that. Even then there are two
arguments against it. Tree models are not troubled by correlated inputs, so
there is little to gain. And PCA mixes the columns into unnamed combinations,
which destroys the ability to say *"the reading 24 hours ago mattered most"* —
one of the more reportable results this project can produce.

Worth revisiting only if we add many more inputs, for example weather data or
several monitoring stations.

# Next

**Part 2 — Prepare the data.** Act on decisions 1 through 6, and build the
inputs the model will learn from.